# RPD04 - Finetuning and Evaluating SmolVLA  (SO-101 - LeRobot - ROCm)

### Lab Description

This lab finetunes **SmolVLA**, a 450M-parameter **vision-language-action** foundation model, on the same dataset you used for ACT in RPD03 -- and compares the two. Unlike ACT, SmolVLA starts from weights pretrained on diverse LeRobot community data and accepts a natural-language instruction. You deploy the base model to see why pretraining alone is not enough, finetune it on *your* demonstrations, evaluate it autonomously on the arm, and weigh the trade-offs between a task-specific model (ACT) and a finetuned foundation model (SmolVLA).

## Lab Overview

| Step | Topic | Key Concepts |
|------|-------|-------------|
| 1 | What is SmolVLA? | Vision-Language-Action model, flow matching, language conditioning |
| 2 | Install SmolVLA Dependencies | Extra Python packages required for SmolVLA |
| 3 | Verify the Environment | PyTorch, ROCm, GPU, SmolVLA imports |
| 4 | Download SmolVLA Base Model | Hugging Face Hub, pretrained weights, config inspection |
| 5 | Experiment: Base Model Without Finetuning | Deploy pretrained SmolVLA directly, observe limitations |
| 6 | Finetune SmolVLA on Your Dataset | `lerobot-train` with local patched base model |
| 7 | Evaluate the Finetuned Model | `lerobot-record` with `--policy.path`, autonomous execution, video playback |
| 8 | ACT vs SmolVLA | Architecture comparison, when to use which |

#### Recommended Hardware

- Two SO-101 arms (one leader, one follower) with Feetech STS3215 servos
- USB hub connecting both arms to the host machine
- Two USB cameras (front and side views)
- GPU with ROCm support (for training and inference)

#### Software Environment

Docker container built from `Dockerfile` with [LeRobot](https://github.com/huggingface/lerobot) v0.5.0+ and ROCm PyTorch.\
Launch with `run.sh` (GPU + USB devices mounted).

#### Prerequisites

- Completed **RPD01** (teleoperation working, calibration verified)
- Completed **RPD02** (dataset recorded at `/opt/workspace/lerobot/local_data/my_dataset`)
- Completed **RPD03** (ACT training and evaluation, for comparison)

## Goals

1. **Understand SmolVLA**: Learn what Vision-Language-Action models are and how SmolVLA differs from ACT.
2. **Finetune a Foundation Model**: Finetune the pretrained SmolVLA base on your demonstration dataset.
3. **Evaluate on the Robot**: Deploy the finetuned model on the follower arm and observe autonomous behavior.
4. **Compare with ACT**: Understand the trade-offs between ACT and SmolVLA for manipulation tasks.

## 1. What is SmolVLA?

**SmolVLA** is Hugging Face's lightweight **Vision-Language-Action (VLA)** foundation model for robotics, introduced in *"SmolVLA: A Vision-Language-Action Model for Affordable and Efficient Robotics"* (2025). With only **450 million parameters**, it is designed to run on consumer-grade hardware while achieving competitive performance against much larger models.

### How SmolVLA Differs from ACT

In RPD03, we trained **ACT (Action Chunking with Transformers)**, which is a task-specific imitation learning algorithm trained from scratch on your demonstrations. SmolVLA takes a fundamentally different approach:

| | ACT (RPD03) | SmolVLA (RPD04) |
|:---|:---|:---|
| **Type** | Task-specific imitation learning | Vision-Language-Action foundation model |
| **Training** | Train from scratch on your data | Finetune a pretrained base model |
| **Language** | No language conditioning | Accepts natural language instructions |
| **Pretraining** | None | Pretrained on diverse LeRobot community datasets |
| **Parameters** | ~197 MB model | ~450M parameters |
| **Action generation** | Transformer decoder + CVAE | Flow Matching Transformer |

### Key Ideas

**Vision-Language-Action (VLA)** -- SmolVLA combines vision understanding, language comprehension, and action generation in a single model. It takes:

- **Camera images** (multiple views)
- **Robot joint state** (current positions)
- **Natural language instruction** (e.g. "Pick up the object")

And outputs a **chunk of future joint actions**, similar to ACT.

**Flow Matching** -- Instead of the CVAE used in ACT, SmolVLA uses a **Flow Matching Transformer** to generate actions. Flow matching learns to transform noise into actions through a continuous flow, producing smooth and diverse action trajectories.

**Foundation Model Approach** -- SmolVLA base (`lerobot/smolvla_base`) is pretrained on thousands of robot demonstrations from the LeRobot community. This gives it a general understanding of robotic manipulation. You then **finetune** it on your specific task and robot, rather than training from scratch.

### Architecture Overview

```
Camera Images -----> [SmolVLM-2 Vision Encoder] -----> Visual Tokens
                                                            |
Language Instruction -> [SmolVLM-2 Language Model] -------->|
                                                            v
Joint Positions ---------> [Projection] -------> [Flow Matching Transformer] ---> Action Chunk
                                                                                   (next K steps)
```

### Why Try SmolVLA?

- **Pretrained knowledge**: The base model already understands general manipulation, potentially requiring fewer demonstrations
- **Language conditioning**: You can describe the task in natural language
- **Competitive performance**: Outperforms ACT on several simulation benchmarks (LIBERO, Meta-World)
- **Efficient**: Designed to run on consumer GPUs

## 2. Install SmolVLA Dependencies

SmolVLA requires additional Python packages (e.g. `transformers`, `accelerate`) that are not included in the base LeRobot installation. Run the cell below to install them and patch a known bug in lerobot 0.5.x.

> **Note:** If you rebuilt the Docker image with the updated `Dockerfile`, these dependencies and the patch are already included. Running the cell again is harmless -- it will simply confirm everything is in place.

In [21]:
import subprocess, sys

print("Installing SmolVLA dependencies...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "transformers>=5,<6", "accelerate>=1.0", "num2words", "--quiet"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("OK")
else:
    print("Install issue (may be fine if already present):")
    print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)

Installing SmolVLA dependencies...
OK


### Patch: GROOT Import Bug (lerobot 0.5.x + transformers 5.x)

LeRobot 0.5.x unconditionally imports the GROOT policy module at startup. With `transformers>=5`, this import crashes due to a dataclass incompatibility -- even if you only use SmolVLA. The cell below patches two files to neutralize the GROOT imports:

- `lerobot/policies/__init__.py` -- comments out the top-level GROOT import
- `lerobot/policies/factory.py` -- wraps the top-level GROOT import in try/except

The patch is **idempotent** (safe to run multiple times) and does not affect any other policy.

> Skip this cell if you rebuilt the Docker image with the updated `Dockerfile` (which includes this patch).

In [22]:
import subprocess, sys, pathlib

SITE = pathlib.Path("/usr/local/lib/python3.12/dist-packages")
MARKER = "# [Lab4-groot-patch]"

def sudo_write(path, content):
    subprocess.run(
        ["sudo", "python3", "-c",
         f"open('{path}','w').write(open('/dev/stdin').read())"],
        input=content, text=True, check=True)

def patch_file(path, content):
    try:
        path.write_text(content)
    except PermissionError:
        sudo_write(path, content)

def patch_init():
    p = SITE / "lerobot/policies/__init__.py"
    if not p.exists():
        return
    text = p.read_text()
    if MARKER in text:
        print("  __init__.py -- OK (already patched)")
        return
    lines = text.splitlines()
    new_lines = [MARKER]
    for line in lines:
        if "GrootConfig" in line and not line.lstrip().startswith("#"):
            new_lines.append("# " + line)
        else:
            new_lines.append(line)
    patch_file(p, "\n".join(new_lines) + "\n")
    print("  __init__.py -- patched")

def patch_factory():
    p = SITE / "lerobot/policies/factory.py"
    if not p.exists():
        return
    text = p.read_text()
    if MARKER in text and "_GrootStub" in text:
        print("  factory.py  -- OK (already patched)")
        return
    if MARKER in text:
        text = text.replace(MARKER + "\n", "")
        text = text.replace("try:\n    from lerobot.policies.groot.configuration_groot import GrootConfig\nexcept Exception:\n    GrootConfig = None",
                            "from lerobot.policies.groot.configuration_groot import GrootConfig")
        patch_file(p, text)
        print("  factory.py  -- removed old None-stub patch, re-patching...")
        text = p.read_text()
    target = "from lerobot.policies.groot.configuration_groot import GrootConfig"
    idx = text.find(target)
    if idx == -1:
        print("  factory.py  -- OK (GROOT import not found)")
        return
    replacement = (
        f"{MARKER}\n"
        f"try:\n"
        f"    {target}\n"
        f"except Exception:\n"
        f"    class _GrootStub: pass\n"
        f"    GrootConfig = _GrootStub"
    )
    text = text[:idx] + replacement + text[idx + len(target):]
    patch_file(p, text)
    print("  factory.py  -- patched")

print("Patching GROOT imports for transformers 5.x compatibility...")
patch_init()
patch_factory()
subprocess.run(["sudo", "find", str(SITE / "lerobot/policies/groot"),
                "-name", "__pycache__", "-exec", "rm", "-rf", "{}", "+"],
               capture_output=True)

for key in list(sys.modules.keys()):
    if "lerobot" in key:
        del sys.modules[key]

try:
    from lerobot.scripts.lerobot_record import main
    print("\nSmolVLA is ready.")
except Exception as e:
    print(f"\nVerification failed: {e}")
    print("Restart the kernel and re-run this cell.")

Overwriting feature type 'VideoFrame' (VideoFrame -> VideoFrame)


Patching GROOT imports for transformers 5.x compatibility...
  __init__.py -- OK (already patched)
  factory.py  -- removed old None-stub patch, re-patching...
  factory.py  -- patched

SmolVLA is ready.


## 3. Verify the Environment

In [23]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU memory: {mem_gb:.1f} GB")

print()
try:
    import transformers
    print(f"transformers: {transformers.__version__}")
except ImportError:
    print("ERROR: transformers not found. Re-run the install cell above.")

try:
    import accelerate
    print(f"accelerate:   {accelerate.__version__}")
except ImportError:
    print("ERROR: accelerate not found. Re-run the install cell above.")

PyTorch version: 2.9.1+rocm7.11.0
CUDA available: True
GPU device: Radeon 8060S Graphics
GPU memory: 61.7 GB

transformers: 5.4.0
accelerate:   1.13.0


## 4. Download SmolVLA Base Model

SmolVLA uses a **finetune-from-base** workflow. Unlike ACT (which is trained from scratch), SmolVLA always starts from a pretrained foundation model and adapts it to your robot and task.

The base model `lerobot/smolvla_base` (450M parameters) is hosted on Hugging Face Hub. We download it here for inspection. During training, lerobot will use this model as the starting point.

In [24]:
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path("/opt/workspace/lerobot/models/smolvla_base")

if not model_dir.exists():
    print("Downloading SmolVLA base model from Hugging Face Hub...")
    print("This may take several minutes (~1.7 GB).\n")
    snapshot_download(
        repo_id="lerobot/smolvla_base",
        local_dir=str(model_dir),
    )
    print(f"\nModel saved to: {model_dir}")
else:
    print(f"Model already exists at: {model_dir}")

print("\nModel files:")
for f in sorted(model_dir.rglob("*")):
    if f.is_file() and not str(f.relative_to(model_dir)).startswith(".cache"):
        size_mb = f.stat().st_size / (1024 * 1024)
        if size_mb > 0.01:
            print(f"  {f.relative_to(model_dir)}  ({size_mb:.1f} MB)")

Model already exists at: /opt/workspace/lerobot/models/smolvla_base

Model files:
  collage_small.gif  (7.6 MB)
  model.safetensors  (864.7 MB)


In [15]:
import json
from pathlib import Path

config_path = Path("/opt/workspace/lerobot/models/smolvla_base/config.json")

if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)

    print("=== SmolVLA Base Model Configuration ===\n")
    print(f"  Policy type:       {config.get('type', 'N/A')}")
    print(f"  Chunk size:        {config.get('chunk_size', 'N/A')}")
    print(f"  n_action_steps:    {config.get('n_action_steps', 'N/A')}")

    input_feats = config.get("input_features", {})
    output_feats = config.get("output_features", {})

    if input_feats:
        print(f"\n  Input features:")
        for key, val in input_feats.items():
            print(f"    {key:40s} -> shape={val.get('shape', 'N/A')}, type={val.get('type', 'N/A')}")

    if output_feats:
        print(f"\n  Output features:")
        for key, val in output_feats.items():
            print(f"    {key:40s} -> shape={val.get('shape', 'N/A')}, type={val.get('type', 'N/A')}")

    print(f"\n  Full config keys: {list(config.keys())}")
else:
    print("config.json not found. Run the download cell above first.")

=== SmolVLA Base Model Configuration ===

  Policy type:       smolvla
  Chunk size:        50
  n_action_steps:    50

  Input features:
    observation.state                        -> shape=[6], type=STATE
    observation.images.camera1               -> shape=[3, 256, 256], type=VISUAL
    observation.images.camera2               -> shape=[3, 256, 256], type=VISUAL
    observation.images.camera3               -> shape=[3, 256, 256], type=VISUAL

  Output features:
    action                                   -> shape=[6], type=ACTION

  Full config keys: ['type', 'n_obs_steps', 'input_features', 'output_features', 'device', 'use_amp', 'push_to_hub', 'repo_id', 'private', 'tags', 'license', 'chunk_size', 'n_action_steps', 'normalization_mapping', 'max_state_dim', 'max_action_dim', 'resize_imgs_with_padding', 'empty_cameras', 'adapt_to_pi_aloha', 'use_delta_joint_actions_aloha', 'tokenizer_max_length', 'num_steps', 'use_cache', 'freeze_vision_encoder', 'train_expert_only', 'train_state

## 5. Experiment: Deploying the Base Model Without Finetuning

In RPD03, the pretrained ACT model (trained on ALOHA) **crashed immediately** on our SO-101 due to hard dimension mismatches (14 joints vs 6 joints).

SmolVLA is different. The base model was pretrained on diverse robot data **including SO-100 style robots** with 6 joints. Looking at the config above, it already expects `observation.state` with shape=[6] and `action` with shape=[6] -- matching our SO-101.

This means we can actually **deploy the base model directly** on our robot. But will it perform the task correctly? Let's find out.

### Patch the Model Config for Our Camera Setup

The SmolVLA base model was pretrained with 3 cameras named `camera1`, `camera2`, `camera3`. Our SO-101 setup uses 2 cameras named `front` and `side`. LeRobot's feature validation requires that the policy's expected camera names match the dataset exactly.

To resolve this, we temporarily patch the local copy of the model's `config.json` so that its `input_features` match our 2-camera setup. SmolVLA processes each image independently through a shared VLM backbone, so using 2 cameras instead of 3 simply means fewer visual tokens -- the model still works correctly.

Run the cell below to apply the patch (a backup of the original config is created automatically).

In [29]:
import json, shutil
from pathlib import Path

model_dir = Path("/opt/workspace/lerobot/models/smolvla_base")
cfg_path = model_dir / "config.json"
bak_path = model_dir / "config.json.bak"

if not cfg_path.exists():
    print(f"ERROR: {cfg_path} not found. Run the download cell above first.")
else:
    if not bak_path.exists():
        shutil.copy2(cfg_path, bak_path)
        print(f"Backed up original config to {bak_path}")

    with open(cfg_path) as f:
        cfg = json.load(f)

    original_features = cfg.get("input_features", {})
    visual_keys = [k for k in original_features if "images" in k]
    print(f"Original visual features: {visual_keys}")

    cfg["input_features"] = {
        "observation.images.front": {"shape": [3, 256, 256], "type": "VISUAL"},
        "observation.images.side":  {"shape": [3, 256, 256], "type": "VISUAL"},
        "observation.state":        {"shape": [6],           "type": "STATE"},
    }

    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)

    patched_keys = [k for k in cfg["input_features"] if "images" in k]
    print(f"Patched visual features:  {patched_keys}")
    print("Config patched for 2-camera setup (front, side).")

Original visual features: ['observation.images.camera1', 'observation.images.camera2', 'observation.images.camera3']
Patched visual features:  ['observation.images.front', 'observation.images.side']
Config patched for 2-camera setup (front, side).


### Deploy the Base Model

Now run the evaluation in a **JupyterLab Terminal** (File -> New -> Terminal):

```bash
rm -rf /opt/workspace/lerobot/local_data/eval_smolvla_base

lerobot-record \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm \
    --robot.calibration_dir=/opt/workspace/lerobot/calibration \
    --robot.cameras='{ front: {type: opencv, index_or_path: /dev/video3, width: 640, height: 480, fps: 30, fourcc: MJPG, warmup_s: 3}, side: {type: opencv, index_or_path: /dev/video1, width: 640, height: 480, fps: 30, fourcc: MJPG, warmup_s: 3}}' \
    --display_data=false \
    --dataset.repo_id=local/eval_smolvla_base \
    --dataset.root=/opt/workspace/lerobot/local_data/eval_smolvla_base \
    --dataset.num_episodes=1 \
    --dataset.single_task="Pick up the object" \
    --dataset.episode_time_s=20 \
    --dataset.reset_time_s=5 \
    --dataset.push_to_hub=false \
    --dataset.vcodec=h264 \
    --dataset.streaming_encoding=true \
    --dataset.encoder_threads=2 \
    --policy.path=/opt/workspace/lerobot/models/smolvla_base \
    --policy.device=cuda \
    --play_sounds=false \
    2>&1 | grep -v "Corrupt JPEG" | grep -v "skipping action generation" | grep -v "running slower"
```

> **Safety:** The base model has never seen your workspace. The robot may move **unpredictably**. Keep your hand near the USB cable and be ready to press **Ctrl+C** or unplug if the motion looks dangerous.

### Visualize Base Model Results

After the evaluation episode completes, run the cell below to watch what the **unfinetuned** base model did.

In [30]:
from pathlib import Path
from IPython.display import display, Video, HTML

eval_dir = Path("/opt/workspace/lerobot/local_data/eval_smolvla_base")
video_dir = eval_dir / "videos"

if not video_dir.exists():
    print(f"No videos found at {video_dir}")
    print("Run the base model evaluation command above first.")
else:
    videos = sorted(video_dir.rglob("*.mp4"))
    if not videos:
        print(f"No .mp4 files found under {video_dir}")
        for d in sorted(video_dir.iterdir()):
            print(f"  {d}")
    else:
        display(HTML("<h3>Base Model (No Finetuning) -- Evaluation Video</h3>"))
        print(f"Found {len(videos)} video(s):\n")
        for v in videos:
            size_mb = v.stat().st_size / (1024 * 1024)
            print(f"  {v.relative_to(video_dir)}  ({size_mb:.1f} MB)")

        for v in videos:
            label = v.relative_to(video_dir)
            display(HTML(f"<h4>{label}</h4>"))
            display(Video(str(v), embed=True, html_attributes="controls loop width=640"))

Found 2 video(s):

  observation.images.front/chunk-000/file-000.mp4  (0.8 MB)
  observation.images.side/chunk-000/file-000.mp4  (1.1 MB)


### Why the Base Model Fails

Unlike ACT (which crashes with a dimension error), the SmolVLA base model **runs without errors** on our SO-101. However, the robot's behavior is poor -- it may move randomly, shake, or do nothing related to the target task. This is because:

| Factor | Impact |
|:---|:---|
| **Task unknown** | The base model has general manipulation knowledge but has never seen your specific workspace, object positions, or task. |
| **Observation statistics** | Joint position ranges and image distributions from your setup differ from the pretraining data. |
| **Camera count** | We patched the config from 3 cameras to 2, so the model receives fewer visual tokens than it was pretrained with. |

> **Key takeaway:** Even though SmolVLA's architecture is compatible with our robot (6 joints, same action space), the model still needs **finetuning on your own demonstrations** to perform the specific task. The advantage over ACT is that finetuning from a pretrained base can converge **faster** and potentially with **fewer demonstrations**.

In the next section, we finetune SmolVLA on our dataset and compare the result.

## 6. Finetune SmolVLA on Your Dataset

Now we finetune the SmolVLA base model on the same demonstration data we collected in RPD02 and used to train ACT in RPD03.

### Verify Dataset

In [31]:
import json
from pathlib import Path

dataset_dir = Path("/opt/workspace/lerobot/local_data/my_dataset")

if dataset_dir.exists():
    info_path = dataset_dir / "meta" / "info.json"
    if info_path.exists():
        with open(info_path) as f:
            info = json.load(f)
        print(f"Dataset:         {dataset_dir}")
        print(f"Total episodes:  {info.get('total_episodes', 'N/A')}")
        print(f"Total frames:    {info.get('total_frames', 'N/A')}")
        print(f"FPS:             {info.get('fps', 'N/A')}")
        print(f"Features:        {list(info.get('features', {}).keys())}")

        tasks = info.get("tasks", {})
        if tasks:
            print(f"Tasks:           {tasks}")
    else:
        print("Dataset directory exists but meta/info.json not found.")
else:
    print(f"Dataset not found at {dataset_dir}")
    print("Please complete Lab 2 first to record a dataset.")

Dataset:         /opt/workspace/lerobot/local_data/my_dataset
Total episodes:  20
Total frames:    17917
FPS:             30
Features:        ['action', 'observation.state', 'observation.images.front', 'observation.images.side', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']


### Run Training

Training must be run in a **JupyterLab Terminal** (File -> New -> Terminal) because it is a long-running process with continuous log output.

```bash
HF_HUB_OFFLINE=1 lerobot-train \
    --policy.path=/opt/workspace/lerobot/models/smolvla_base \
    --dataset.repo_id=local/my_dataset \
    --dataset.root=/opt/workspace/lerobot/local_data/my_dataset \
    --dataset.video_backend=pyav \
    --batch_size=8 \
    --steps=20000 \
    --output_dir=/opt/workspace/lerobot/models/smolvla_my_dataset \
    --job_name=smolvla_my_dataset \
    --policy.device=cuda \
    --policy.push_to_hub=false \
    --wandb.enable=false
```

> **Important:** We use the **local path** (`/opt/workspace/lerobot/models/smolvla_base`) instead of the Hub repo ID (`lerobot/smolvla_base`). The local copy has been patched (in Section 5) to use our 2-camera setup (`front`, `side`). Make sure the config patch cell has been run before training.

### Training Parameters

| Parameter | Description | Value |
|:---|:---|:---|
| `--policy.path` | Pretrained base model to finetune from | `/opt/workspace/lerobot/models/smolvla_base` (local, config patched for 2 cameras) |
| `--dataset.repo_id` | Dataset identifier (logical name) | `local/my_dataset` |
| `--dataset.root` | Path to the local dataset from RPD02 | `/opt/workspace/lerobot/local_data/my_dataset` |
| `--dataset.video_backend` | Video decoder backend | `pyav` (default `torchcodec` has ABI issues with ROCm) |
| `--batch_size` | Training batch size | `8` (increase if GPU memory allows; A100 uses 64) |
| `--steps` | Total training steps | `20000` (SmolVLA recommended; adjust based on performance) |
| `--output_dir` | Where to save checkpoints and logs | `/opt/workspace/lerobot/models/smolvla_my_dataset` |
| `--policy.device` | Device for training | `cuda` |
| `--policy.push_to_hub` | Upload trained model to Hub | `false` |
| `--wandb.enable` | Enable Weights & Biases logging | `false` |

### Key Differences from ACT Training (RPD03)

| | ACT (RPD03) | SmolVLA (RPD04) |
|:---|:---|:---|
| **Starting point** | `--policy.type=act` (random init) | `--policy.path=.../smolvla_base` (pretrained, local) |
| **Batch size** | Default (varies) | `8` (explicit, tunable) |
| **Steps** | 30,000+ | 20,000 (converges faster due to pretraining) |
| **Output directory** | `models/act_my_dataset` | `models/smolvla_my_dataset` |

> **Note:** Do NOT pass `--policy.type=smolvla` when using `--policy.path`. The policy type is automatically inferred from the base model's `config.json`.

> **Memory Tip:** If you get an out-of-memory (OOM) error, reduce `--batch_size` to 4 or even 2. SmolVLA (450M params) uses more GPU memory than ACT.

### Monitoring Training Progress

Training logs appear in the terminal. Key metrics to watch:

```
step: 1000  loss: 0.0452  grad_norm: 12.3  lr: 1e-05  ...
step: 2000  loss: 0.0321  grad_norm: 8.7   lr: 1e-05  ...
```

- **loss** -- Should decrease over time. SmolVLA typically starts with lower loss than ACT (thanks to pretraining).
- **grad_norm** -- Gradient magnitude. Should remain stable.
- **lr** -- Learning rate.

Checkpoints are saved periodically to `--output_dir`. The final model is at:

```
/opt/workspace/lerobot/models/smolvla_my_dataset/checkpoints/last/pretrained_model/
```

### Resuming Interrupted Training

If training is interrupted, resume from the last checkpoint:

```bash
lerobot-train \
    --policy.path=/opt/workspace/lerobot/models/smolvla_base \
    --dataset.repo_id=local/my_dataset \
    --dataset.root=/opt/workspace/lerobot/local_data/my_dataset \
    --dataset.video_backend=pyav \
    --batch_size=8 \
    --steps=20000 \
    --output_dir=/opt/workspace/lerobot/models/smolvla_my_dataset \
    --job_name=smolvla_my_dataset \
    --policy.device=cuda \
    --policy.push_to_hub=false \
    --wandb.enable=false \
    --resume=true
```

### Verify Training Output

After training completes, verify that model files were saved.

In [32]:
from pathlib import Path

model_dir = Path("/opt/workspace/lerobot/models/smolvla_my_dataset")
last_ckpt = model_dir / "checkpoints" / "last" / "pretrained_model"

if last_ckpt.exists():
    print(f"Finetuned SmolVLA model found at: {last_ckpt}\n")
    for f in sorted(last_ckpt.rglob("*")):
        if f.is_file():
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  {f.relative_to(last_ckpt)}  ({size_mb:.1f} MB)")
else:
    print(f"Model not found at {last_ckpt}")
    print("Make sure training has completed in the terminal.")

    ckpt_dir = model_dir / "checkpoints"
    if ckpt_dir.exists():
        ckpts = sorted(ckpt_dir.iterdir())
        if ckpts:
            print(f"\nAvailable checkpoints: {[c.name for c in ckpts]}")

Finetuned SmolVLA model found at: /opt/workspace/lerobot/models/smolvla_my_dataset/checkpoints/last/pretrained_model

  config.json  (0.0 MB)
  model.safetensors  (864.7 MB)
  policy_postprocessor.json  (0.0 MB)
  policy_postprocessor_step_0_unnormalizer_processor.safetensors  (0.0 MB)
  policy_preprocessor.json  (0.0 MB)
  policy_preprocessor_step_5_normalizer_processor.safetensors  (0.0 MB)
  train_config.json  (0.0 MB)


## 7. Evaluate the Finetuned Model

Deploy the finetuned SmolVLA on the real robot. The follower arm will execute actions **autonomously** -- no leader arm or human teleoperation is needed.

### Run Evaluation

Run the following in a **JupyterLab Terminal**:

```bash
rm -rf /opt/workspace/lerobot/local_data/eval_smolvla

lerobot-record \
    --robot.type=so101_follower \
    --robot.port=/dev/ttyACM0 \
    --robot.id=my_follower_arm \
    --robot.calibration_dir=/opt/workspace/lerobot/calibration \
    --robot.cameras='{ front: {type: opencv, index_or_path: /dev/video3, width: 640, height: 480, fps: 30, fourcc: MJPG, warmup_s: 3}, side: {type: opencv, index_or_path: /dev/video1, width: 640, height: 480, fps: 30, fourcc: MJPG, warmup_s: 3}}' \
    --display_data=false \
    --dataset.repo_id=local/eval_smolvla \
    --dataset.root=/opt/workspace/lerobot/local_data/eval_smolvla \
    --dataset.num_episodes=1 \
    --dataset.single_task="Pick up the object" \
    --dataset.episode_time_s=15 \
    --dataset.reset_time_s=5 \
    --dataset.push_to_hub=false \
    --dataset.vcodec=h264 \
    --dataset.streaming_encoding=true \
    --dataset.encoder_threads=2 \
    --policy.path=/opt/workspace/lerobot/models/smolvla_my_dataset/checkpoints/last/pretrained_model \
    --policy.device=cuda \
    --play_sounds=false \
    2>&1 | grep -v "Corrupt JPEG" | grep -v "skipping action generation" | grep -v "running slower"
```

> **Important:** The `--dataset.single_task` should match the task description used during data collection in RPD02. SmolVLA uses this as the **language instruction** to condition its action predictions.

> **Important:** Camera key names (`front`, `side`) must match **exactly** what was used during data collection in RPD02.

### Key Parameters

| Parameter | Value | Why |
|:---|:---|:---|
| `--dataset.num_episodes=1` | 1 episode per run | The gripper won't release between episodes. Run one at a time, then manually reset. |
| `--dataset.episode_time_s=10` | 10 seconds | Match the typical duration of your training demonstrations. |
| `--dataset.reset_time_s=1` | 1 second | Minimal reset since we run one episode per run. |
| `--dataset.vcodec=h264` | H.264 codec | Much faster than default AV1 encoding. |
| `--dataset.streaming_encoding=true` | Encode in parallel | Avoids post-episode encoding delay. |

### What to Expect

- The robot will **move on its own** based on SmolVLA's predictions
- Each episode runs for `episode_time_s` seconds, then stops
- The robot may freeze after completing its learned motion -- this is normal
- If the gripper locks: press **Ctrl+C** first, then unplug the follower's USB cable to release
- Evaluation episodes are recorded to `/opt/workspace/lerobot/local_data/eval_smolvla/`

### Visualize Evaluation Results

After evaluation, run the cell below to play the recorded videos directly in this notebook.

In [34]:
from pathlib import Path
from IPython.display import display, Video, HTML

eval_dir = Path("/opt/workspace/lerobot/local_data/eval_smolvla")
video_dir = eval_dir / "videos"

if not video_dir.exists():
    print(f"No videos found at {video_dir}")
    print("Run the evaluation command above first.")
else:
    videos = sorted(video_dir.rglob("*.mp4"))
    if not videos:
        print(f"No .mp4 files found under {video_dir}")
        print("Subdirectories found:")
        for d in sorted(video_dir.iterdir()):
            print(f"  {d}")
    else:
        print(f"Found {len(videos)} evaluation video(s):\n")
        for v in videos:
            size_mb = v.stat().st_size / (1024 * 1024)
            print(f"  {v.relative_to(video_dir)}  ({size_mb:.1f} MB)")

        for v in videos:
            label = v.relative_to(video_dir)
            display(HTML(f"<h4>{label}</h4>"))
            display(Video(str(v), embed=True, html_attributes="controls loop width=640"))

Found 2 evaluation video(s):

  observation.images.front/chunk-000/file-000.mp4  (0.6 MB)
  observation.images.side/chunk-000/file-000.mp4  (0.9 MB)


## 8. ACT vs SmolVLA: When to Use Which

Having trained and evaluated both ACT (RPD03) and SmolVLA (RPD04) on the same dataset, you can now compare their performance side-by-side.

### Architecture Comparison

| Aspect | ACT | SmolVLA |
|:---|:---|:---|
| **Model type** | Task-specific imitation learning | Vision-Language-Action foundation model |
| **Pretraining** | None (trained from scratch) | Pretrained on diverse LeRobot community data |
| **Language input** | No | Yes (natural language instruction) |
| **Action generation** | CVAE + Transformer decoder | Flow Matching Transformer |
| **Model size** | ~197 MB | ~1.7 GB |
| **Training time** | Faster per step | Slower per step (larger model) |
| **Data efficiency** | Needs 50+ demonstrations | Potentially fewer (leverages pretraining) |
| **GPU memory** | Lower (~4-8 GB) | Higher (~8-16 GB) |

### When to Use ACT

- **Limited GPU memory** -- ACT is much smaller and trains faster
- **Simple, repetitive tasks** -- When the task doesn't change and you have enough demonstrations
- **Quick iteration** -- Faster training cycle for experimentation
- **No language conditioning needed** -- Task is always the same

### When to Use SmolVLA

- **Language-conditioned tasks** -- When you want to specify different tasks via language
- **Limited demonstrations** -- SmolVLA's pretraining may help it learn from fewer examples
- **Multi-task learning** -- One model can potentially handle multiple tasks via different instructions
- **Transfer learning** -- Foundation model knowledge may transfer to new scenarios

### Improving Results for Both

| Strategy | Details |
|:---|:---|
| **Collect more data** | Both benefit from more demonstrations. 50-100+ episodes recommended. |
| **Improve data quality** | Consistent demonstrations with similar starting poses and strategies. |
| **Train longer** | ACT: 50,000-100,000 steps. SmolVLA: 20,000-40,000 steps. |
| **Adjust batch size** | Larger batch sizes generally improve training stability. |
| **Tune hyperparameters** | Chunk size, learning rate, and model-specific parameters. |

### Conclusion

In this lab, you have:

- Learned what **SmolVLA (Vision-Language-Action)** models are and how they differ from ACT
- Installed the required SmolVLA dependencies and downloaded the **pretrained base model**
- **Finetuned SmolVLA** on your own demonstration dataset (the same data used for ACT in RPD03)
- **Evaluated the finetuned model** on the real robot in autonomous mode
- **Compared ACT and SmolVLA** to understand their respective strengths and trade-offs

The key insight is that SmolVLA represents a **foundation model approach** to robot learning -- rather than training from scratch each time, you leverage a pretrained model's general manipulation knowledge and adapt it to your specific setup. Whether SmolVLA or ACT works better depends on your task complexity, available data, and compute resources. Having both tools in your toolbox gives you flexibility for different scenarios.

A natural next step is **cross-task generalization** -- training a single SmolVLA model on multiple manipulation tasks and switching behaviors at inference time using only the language instruction.

## Acknowledgements

This lab series builds on the [LeRobot](https://github.com/huggingface/lerobot) project -- the `lerobot-teleoperate`, `lerobot-record`, and `lerobot-train` tools, the **ACT** and **SmolVLA** policies -- and the open-hardware **SO-101** arm from [TheRobotStudio / Hugging Face](https://github.com/TheRobotStudio/SO-ARM100). Training and inference run on AMD Ryzen AI (Radeon gfx1152) with ROCm.


---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
